## Vehicle Routing Problem using OR-Tools
### Import the required libraries

In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import matplotlib as mpl
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

#------
from sklearn.metrics.pairwise import haversine_distances                                                          
from math import radians   

### Load the problem data from file.
Load the CSV file and store it into a dataframe. Set the <code>coordinates</code> from the <code>x</code> and <code>y</code> columns. Set the <code>demands</code> from the <code>d</code> column.


In [2]:
path = pd.read_csv('D:/Ateneo de Davao/College/4thYear1stSem/Computational Optimization & Decision Making/Applied-Combinatorial-Optimisation-code/lab_6_VRP_OR_TOOLS/X-n200-k36.csv')
path

,Unnamed: 0,x,y,d
0,0,0,0,0
1,1,957,135,0
2,2,149,141,83
3,3,1,96,52
4,4,228,268,73
...,...,...,...,...
196,196,186,373,66
197,197,300,26,88
198,198,52,204,74
199,199,163,305,70


In [3]:
coordinates = path.loc[:, ['x', 'y']]
demands = path.d.values

coordinates.head()

,x,y
0,0,0
1,957,135
2,149,141
3,1,96
4,228,268


#### Set the parameters: capacity, number of vehicles, and $N$.

In [4]:
capacity = 2000
num_vehicles = 36
N = len(coordinates)

#### Compute the distance matrix using <code>squareform</code>. 

In [5]:
distances = squareform(pdist(coordinates, metric="euclidean"))
distances = np.round(distances * 1e4, decimals=0).astype(int)

### Instantiate a routing manager and a model.

In [6]:
manager = pywrapcp.RoutingIndexManager(N, num_vehicles, 0)  # < ---- assuming depot is at index 0
routing = pywrapcp.RoutingModel(manager)

### Define the distance callback.

In [7]:
def distance_callback(from_index, to_index):
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distances[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

### Define the demand callback. 

In [8]:
def demand_callback(from_index):
    # Convert from routing variable Index to demands NodeIndex.
    from_node = manager.IndexToNode(from_index)
    return demands[from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

### Add Constraints
#### Add dimensions related to arcs/weights.

In [9]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  
    [capacity,] * num_vehicles,  
    True,  
    'Capacity'
)

True

### Set the search strategies
Use the default routing search parameters. Use the <code>PATH_CHEAPEST_ARC</code> as the strategy to look for the first solution. 

In [20]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC #or SAVINGS or PATH_CHEAPEST_ARC
)

#### Include Tabu Search as a metaheuristic
Set another attribute in the search parameters using the <code>local_search_metaheuristic</code> object.

In [22]:
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.TABU_SEARCH
)

search_parameters.time_limit.seconds = 30

### Set the objective function
The objective function minimises the total transportation cost. Thus, the <code>transit_callback_index</code> is passed to the <code>SetArcCostEvaluatorOfAllVehicles</code> function.

In [23]:
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

### Solve the model
#### Print the solution

In [24]:
def print_solution(data, manager, routing, solution):                                                                     
    print(f"Objective: {solution.ObjectiveValue()}")                                                                      
    max_route_distance = 0                                                                                                
                                                                                                                            
    for vehicle_id in range(num_vehicles):                                                                                
        # Skip vehicles that aren't used in the solution                                                                  
        if not routing.IsVehicleUsed(solution, vehicle_id):                                                               
            continue                                                                                                      
                                                                                                                            
        index = routing.Start(vehicle_id)                                                                                 
        plan_output = f"Route for vehicle {vehicle_id}:\n"                                                                
        route_distance = 0                                                                                                
                                                                                                                            
        while not routing.IsEnd(index):                                                                                   
            plan_output += f" {manager.IndexToNode(index)} -> "                                                           
            previous_index = index                                                                                        
            index = solution.Value(routing.NextVar(index))                                                                
                                                                                                                            
            # Retrieve the distance traveled on this arc                                                                  
            route_distance += routing.GetArcCostForVehicle(                                                               
                previous_index, index, vehicle_id                                                                         
            )                                                                                                             
                                                                                                                            
        plan_output += f"{manager.IndexToNode(index)}\n"                                                                  
        plan_output += f"Distance of the route: {route_distance}m\n"                                                      
        print(plan_output)                                                                                                
                                                                                                                            
        max_route_distance = max(route_distance, max_route_distance)                                                      
                                                                                                                            
    print(f"Maximum of the route distances: {max_route_distance}m")            

In [25]:
print("PLS W8 SIR)")                                                               
solution = routing.SolveWithParameters(search_parameters)                                                                 
                                                                                                                            
if solution:                                                                                                              
    print_solution(distances, manager, routing, solution)                                                               
else:                                                                                                                     
    print("No solution found !") 

PLS W8 SIR)
Objective: 134051464
Route for vehicle 0:
 0 ->  166 ->  91 ->  175 ->  161 ->  113 ->  145 ->  23 ->  190 ->  52 ->  34 ->  16 ->  72 ->  156 ->  4 ->  117 ->  106 ->  51 ->  36 ->  199 ->  94 ->  105 ->  96 ->  165 ->  125 ->  100 -> 0
Distance of the route: 9618752m

Route for vehicle 28:
 0 ->  182 ->  169 ->  158 ->  127 ->  61 ->  48 ->  138 ->  97 ->  57 ->  38 ->  200 ->  185 ->  9 ->  177 ->  176 ->  144 ->  142 ->  37 ->  101 -> 0
Distance of the route: 6905023m

Route for vehicle 29:
 0 ->  103 ->  170 ->  172 ->  30 ->  92 ->  137 ->  159 ->  112 ->  168 ->  18 ->  15 ->  146 ->  73 ->  66 ->  89 ->  107 ->  86 ->  28 ->  130 ->  110 ->  5 ->  151 ->  143 ->  56 ->  27 ->  187 ->  119 ->  24 ->  60 ->  120 ->  173 ->  62 -> 0
Distance of the route: 24360514m

Route for vehicle 30:
 0 ->  50 ->  197 ->  1 ->  93 ->  77 ->  41 ->  126 ->  79 ->  155 ->  122 ->  19 ->  11 ->  32 ->  33 ->  21 ->  13 ->  179 ->  64 ->  59 ->  49 ->  114 ->  6 ->  47 ->  58 ->  152 -